# Pipeline — Landing Zone → Bronze (Delta Lake)

Lê os CSVs gravados no bucket `landing-zone` pelo notebook 01 e persiste no bucket `bronze` no formato Delta Lake, com particionamento onde aplicável.

## 1. Configuração

In [1]:
MINIO_ENDPOINT   = "http://localhost:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
LANDING_BUCKET   = "s3a://landing-zone"
BRONZE_BUCKET    = "s3a://bronze"

# partition_by: coluna(s) para particionar (None = sem particionamento)
TABLES = [
    {"name": "brands",       "partition_by": None},
    {"name": "categories",   "partition_by": None},
    {"name": "customers",    "partition_by": ["state"]},
    {"name": "stores",       "partition_by": None},
    {"name": "staffs",       "partition_by": None},
    {"name": "products",     "partition_by": ["model_year"]},
    {"name": "stocks",       "partition_by": None},
    {"name": "orders",       "partition_by": ["order_status"]},
    {"name": "order_items",  "partition_by": None},
]

## 2. SparkSession com suporte a S3A e Delta Lake

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("landing_to_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    ],
).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession iniciada — versão: {spark.version}")

your 131072x1 screen size is bogus. expect trouble
26/05/06 06:54:37 WARN Utils: Your hostname, Ismael resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 06:54:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/ismasel/projects/eng-de-dados/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ismasel/.ivy2/cache
The jars for the packages stored in: /home/ismasel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b655c564-94a0-4fbf-af8e-379f6143e908;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 229ms :: artifacts dl 14ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr

SparkSession iniciada — versão: 3.5.0


## 3. Leitura dos CSVs do landing-zone

In [3]:
from pyspark.sql.functions import current_timestamp, lit

dataframes = {}
for table in TABLES:
    name = table["name"]
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{LANDING_BUCKET}/{name}")
        .withColumn("bronze_ingested_at", current_timestamp())
    )
    dataframes[name] = df
    print(f"{name}: {df.count()} registros")

26/05/06 06:54:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


brands: 9 registros
categories: 7 registros
customers: 1445 registros
stores: 3 registros
staffs: 10 registros
products: 321 registros
stocks: 939 registros
orders: 1615 registros
order_items: 4722 registros


## 4. Gravação em Delta Lake no bucket bronze

In [4]:
for table in TABLES:
    name         = table["name"]
    partition_by = table["partition_by"]
    df           = dataframes[name]
    output_path  = f"{BRONZE_BUCKET}/{name}"

    writer = df.write.format("delta").mode("overwrite")

    if partition_by:
        writer = writer.partitionBy(*partition_by)

    writer.save(output_path)
    print(f"Delta gravado: {output_path}")

print("\nCarga para a camada bronze concluída.")

26/05/06 06:55:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta gravado: s3a://bronze/brands
Delta gravado: s3a://bronze/categories
Delta gravado: s3a://bronze/customers
Delta gravado: s3a://bronze/stores
Delta gravado: s3a://bronze/staffs
Delta gravado: s3a://bronze/products
Delta gravado: s3a://bronze/stocks
Delta gravado: s3a://bronze/orders
Delta gravado: s3a://bronze/order_items

Carga para a camada bronze concluída.


## 5. Registro das tabelas no catálogo Spark

In [5]:
for table in TABLES:
    name        = table["name"]
    delta_path  = f"{BRONZE_BUCKET}/{name}"
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze_{name}
        USING DELTA
        LOCATION '{delta_path}'
    """)
    print(f"Tabela bronze_{name} registrada no catálogo")

print("\nValidação de contagens:")
for table in TABLES:
    name  = table["name"]
    count = spark.sql(f"SELECT COUNT(*) AS n FROM bronze_{name}").collect()[0]["n"]
    print(f"  bronze_{name}: {count} registros")

Tabela bronze_brands registrada no catálogo
Tabela bronze_categories registrada no catálogo
Tabela bronze_customers registrada no catálogo
Tabela bronze_stores registrada no catálogo
Tabela bronze_staffs registrada no catálogo
Tabela bronze_products registrada no catálogo
Tabela bronze_stocks registrada no catálogo
Tabela bronze_orders registrada no catálogo
Tabela bronze_order_items registrada no catálogo

Validação de contagens:
  bronze_brands: 9 registros
  bronze_categories: 7 registros
  bronze_customers: 1445 registros
  bronze_stores: 3 registros
  bronze_staffs: 10 registros
  bronze_products: 321 registros
  bronze_stocks: 939 registros
  bronze_orders: 1615 registros
  bronze_order_items: 4722 registros


In [6]:
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
